## Benchmarking Machine Learning Approaches for Energy Prediction

### Objective
This project systematically benchmarks different machine learning modelling approaches for predicting household energy consumption.

### 01_EDA
This notebook focuses on:
- Understanding the dataset
- Exploring feature distributions
- Identifying data quality issues

In [1]:
import sys
print(sys.executable)

d:\Serious\Masters\Research\energy-prediction-benchmark\venv\Scripts\python.exe


In [1]:
# Core libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Settings
pd.set_option('display.max_columns', None)
sns.set(style="whitegrid")

In [3]:
# Load datasets
train_df = pd.read_csv("../data/train.csv")
test_df = pd.read_csv("../data/test.csv")
synthetic_df = pd.read_csv("../data/synthetic_data.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Synthetic shape:", synthetic_df.shape)

Train shape: (90000, 7)
Test shape: (390, 5)
Synthetic shape: (390, 5)


In [6]:
print("Training Data columns: \n",train_df.columns.to_list())
print("Test Data columns: \n",test_df.columns.to_list())
print("Synthetic Data columns: \n",synthetic_df.columns.to_list())


Training Data columns: 
 ['Household_ID', 'Date', 'Energy_Consumption_kWh', 'Household_Size', 'Avg_Temperature_C', 'Has_AC', 'Peak_Hours_Usage_kWh']
Test Data columns: 
 ['Household_ID', 'Date', 'County', 'Has_AC', 'Avg_Temperature_C']
Synthetic Data columns: 
 ['Household_ID', 'Date', 'County', 'Energy_Consumption_kWh', 'Has_AC']


In [8]:
train_df.describe()

,Energy_Consumption_kWh
count,390.000000
mean,3.726054
std,5.030881
min,0.010000
25%,0.352500
50%,2.022500
75%,4.987500
max,21.230000


## **Data Preparation Pipeline (Train & Test)**

#### 1. Define Columns and setup

In [9]:
target_col = "Energy_Consumption_kWh"
id_col = "Household_ID"
date_col = "Date"

#### 2. Convert Date and Extract Features

In [10]:
def process_date(df):
    df[date_col] = pd.to_datetime(df[date_col])
    df["month"] = df[date_col].dt.month
    df["day_of_week"] = df[date_col].dt.dayofweek
    df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
    return df

train_df = process_date(train_df)
test_df = process_date(test_df)

C:\Users\khush\AppData\Local\Temp\ipykernel_19028\3050615551.py:2: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[date_col] = pd.to_datetime(df[date_col])


#### 3. Adding and removing columns

In [11]:
# As per assumption: test households = size 1
test_df["Household_Size"] = 1

In [12]:
# Drop 'County' from test since it is not available in train
test_df = test_df.drop(columns=["County"])

In [13]:
features = [
    "Avg_Temperature_C",
    "Has_AC",
    "Household_Size",
    "month",
    "day_of_week",
    "is_weekend"
]

#### 4. Train and Test Matrices

In [14]:
X_train = train_df[features]
y_train = train_df[target_col]

X_test = test_df[features]

In [15]:
# Sanity checks for consistency
print("Train features shape:", X_train.shape)
print("Test features shape:", X_test.shape)

print("\nFeature columns match:",
      list(X_train.columns) == list(X_test.columns))

print("\nMissing values in train:\n", X_train.isnull().sum())
print("\nMissing values in test:\n", X_test.isnull().sum())

Train features shape: (90000, 6)
Test features shape: (390, 6)

Feature columns match: True

Missing values in train:
 Avg_Temperature_C    0
Has_AC               0
Household_Size       0
month                0
day_of_week          0
is_weekend           0
dtype: int64

Missing values in test:
 Avg_Temperature_C    37
Has_AC                0
Household_Size        0
month                 0
day_of_week           0
is_weekend            0
dtype: int64
